In [1]:
import glob
import os
import time
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from time import perf_counter

### Configuration

In [2]:
CHANNEL_CONFIG = "g-a"   # gyro + accel channels

TRAIN_DIR = [R"dataset3"]
TEST_DIR  = [R"dataset3"]

LABEL_MAPPING = {
    'HIT': 0, 'SHAKE': 1, 'SWING_LEFT': 2, 'SWING_RIGHT': 3,
    'FAN': 4, 'IDLE': 5, 'STIR': 6, 'LIFT': 7, 'SPIN': 8, 'POUR': 9
}
CLASS_NAMES = list(LABEL_MAPPING.keys())

# List Model
# package_used: 0 = sklearn  |  1 = sktime  |  2 = aeon
MODELS_TO_RUN = [
    ("Random Forest",   0, "rf"),
    ("SVM",             0, "svm"),
    ("Softmax (LR)",    0, "softmax"),
    ("KNN-DTW",         1, "knn"),
    ("MiniRocket",      1, "minirocket"),
    ("LITEMVTime",      2, "litemvtime"),
]

### Load Data

In [3]:
def build_df(dirs):
    all_csv = []
    for d in dirs:
        csvs = glob.glob(os.path.join(d, "*.csv"))
        print(f"{d} -> {len(csvs)} CSV files")
        all_csv.extend(csvs)
    print(f"Total: {len(all_csv)} CSV files")
    if not all_csv:
        raise ValueError("No CSV files found.")
    df = pd.concat((pd.read_csv(f) for f in all_csv), ignore_index=True)
    df.drop_duplicates(inplace=True)
    return df

df_train = build_df(TRAIN_DIR)
df_test  = build_df(TEST_DIR)
print(f"Train: {len(df_train)} rows | Test: {len(df_test)} rows")

dataset3 -> 90 CSV files
Total: 90 CSV files
dataset3 -> 90 CSV files
Total: 90 CSV files
Train: 8951 rows | Test: 8951 rows


### Feature Engineering

In [4]:
CHANNEL_REGEX = {
    "gyro":  r'^(gyro|motion)',
    "accel": r'^(acc|motion)',
    "mag":   r'^(mag|motion)',
    "ahrs":  r'^(ahrs|motion)',
    "g-a":   r'^(gyro|acc|motion)',
    "g-m":   r'^(gyro|mag|motion)',
    "a-m":   r'^(acc|mag|motion)',
    "g-a-m": r'^(gyro|acc|mag|motion)',
}

CHANNEL_LISTS = {
    "accel":  ["acc_x", "acc_y", "acc_z"],
    "gyro":   ["gyro_x",  "gyro_y",  "gyro_z"],
    "mag":    ["mag_x",   "mag_y",   "mag_z"],
    "ahrs":   ["ahrs_x",  "ahrs_y",  "ahrs_z", "ahrs_w"],
    "g-a":    ["gyro_x",  "gyro_y",  "gyro_z",  "acc_x", "acc_y", "acc_z"],
    "g-m":    ["gyro_x",  "gyro_y",  "gyro_z",  "mag_x",   "mag_y",   "mag_z"],
    "a-m":    ["acc_x", "acc_y", "acc_z", "mag_x",   "mag_y",   "mag_z"],
    "g-a-m":  ["gyro_x",  "gyro_y",  "gyro_z",  "acc_x", "acc_y", "acc_z",
               "mag_x",   "mag_y",   "mag_z"],
}

def filter_channels(df, config):
    regex = CHANNEL_REGEX.get(config)
    return df.filter(regex=regex) if regex else df

def get_X_y(df, package_used, config, n_timesteps=30):
    y = np.vectorize(LABEL_MAPPING.get)(df["motion_type"].values)

    if package_used == 0:  # sklearn – flat feature vector
        df_f = filter_channels(df, config)
        X = df_f.drop(columns=["motion_type"]).values.astype("float32")
        return X, y

    # sktime / aeon – 3-D array (samples, channels, timesteps)
    channels = CHANNEL_LISTS.get(config, list(CHANNEL_LISTS.values())[-1])
    n_channels = len(channels)
    X = np.zeros((len(df), n_channels, n_timesteps), dtype="float32")
    missing = []
    for t in range(n_timesteps):
        for c, ch in enumerate(channels):
            col = f"{ch}_{t}"
            if col not in df.columns:
                missing.append(col)
                continue
            X[:, c, t] = df[col].values
    if missing:
        raise KeyError(
            f"Missing {len(missing)} column(s) in DataFrame.\n"
            f"First few: {missing[:5]}\n"
            f"Available sample columns: {[c for c in df.columns if '_0' in c][:10]}"
        )
    return X, y

### Train

In [5]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from sktime.classification.distance_based import KNeighborsTimeSeriesClassifier
from aeon.classification.deep_learning import LITETimeClassifier

def build_clf(approach, package_used):
    ap = approach.lower()
    if package_used == 0:
        return {"rf": RandomForestClassifier(),
                "svm": SVC(),
                "softmax": LogisticRegression(max_iter=1000)}[ap]
    if package_used == 1:
        return {"knn": KNeighborsTimeSeriesClassifier(n_neighbors=1, distance="dtw"),
                "minirocket": make_pipeline(
                    MiniRocketMultivariate(random_state=42),
                    RidgeClassifier(alpha=1.0)
                )}[ap]
    if package_used == 2:
        return {"litemvtime": LITETimeClassifier(use_litemv=True)}[ap]
    raise ValueError(f"Unknown approach: {approach}")

In [6]:
INFERENCE_REPEATS = 50    # average inference time over N runs

results = []   # list of dicts, one per model

for name, pkg, approach in MODELS_TO_RUN:
    print(f"\n{'='*60}")
    print(f" Training: {name}")
    print(f"{'='*60}")

    X_train, y_train = get_X_y(df_train, pkg, CHANNEL_CONFIG)
    X_test,  y_test  = get_X_y(df_test,  pkg, CHANNEL_CONFIG)

    clf = build_clf(approach, pkg)

    t_fit_start = perf_counter()
    clf.fit(X_train, y_train)
    t_fit_end = perf_counter()
    fit_time = t_fit_end - t_fit_start

    y_pred = clf.predict(X_test)

    # Single-sample inference time
    sample = X_train[0:1]
    times = []
    for _ in range(INFERENCE_REPEATS):
        t0 = perf_counter()
        clf.predict(sample)
        t1 = perf_counter()
        times.append(t1 - t0)
    inf_time_ms = np.mean(times) * 1000  # convert to ms

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, zero_division=0)

    print(f"  Fit time      : {fit_time:.2f}s")
    print(f"  Inference time: {inf_time_ms:.3f} ms  (avg over {INFERENCE_REPEATS} runs)")
    print(f"  Accuracy      : {acc:.4f}")
    print(f"  Precision (W) : {prec:.4f}")
    print(f"  Recall (W)    : {rec:.4f}")
    print(f"  F1 (W)        : {f1:.4f}")

    results.append({
        "Model":          name,
        "Package":        ["sklearn", "sktime", "aeon"][pkg],
        "Fit Time (s)":   round(fit_time, 3),
        "Inf Time (ms)":  round(inf_time_ms, 4),
        "Accuracy":       round(acc,  4),
        "Precision":      round(prec, 4),
        "Recall":         round(rec,  4),
        "F1":             round(f1,   4),
        "confusion_matrix": cm,
        "report":         report,
        "y_test":         y_test,
        "y_pred":         y_pred,
    })

print("\n All models have been trained")


 Training: Random Forest
  Fit time      : 9.46s
  Inference time: 3.025 ms  (avg over 50 runs)
  Accuracy      : 1.0000
  Precision (W) : 1.0000
  Recall (W)    : 1.0000
  F1 (W)        : 1.0000

 Training: SVM
  Fit time      : 1.12s
  Inference time: 0.532 ms  (avg over 50 runs)
  Accuracy      : 0.9939
  Precision (W) : 0.9939
  Recall (W)    : 0.9939
  F1 (W)        : 0.9939

 Training: Softmax (LR)
  Fit time      : 5.47s
  Inference time: 0.052 ms  (avg over 50 runs)
  Accuracy      : 0.8817
  Precision (W) : 0.8825
  Recall (W)    : 0.8817
  F1 (W)        : 0.8820

 Training: KNN-DTW


KeyboardInterrupt: 